# Phase Space Analysis 5D

This notebook mirrors `phase_space_analysis_5d.py` and performs multivariate phase space analysis for 5D vector time series data.

## 1) Imports and configuration

Set the CSV path (default `data/vectors_5d.csv`) and output directory before running the pipeline.

In [ ]:
import os

from phase_space_analysis_5d import (
    AnalysisResults5D,
    build_recurrence_matrix,
    compute_pca,
    compute_rqa_metrics,
    find_poincare_crossings,
    interpret_lle,
    load_and_normalize,
    print_summary,
    rosenstein_lle,
    save_lyapunov_plot,
    save_pca_attractor_2d,
    save_pca_attractor_3d,
    save_poincare_section,
    save_recurrence_plot,
    synthesize_verdict,
)

csv_path = 'data/vectors_5d.csv'
output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

## 2) Data loading and preprocessing

Load the 5D vectors, verify the input shape, and normalize each dimension to zero mean and unit variance.

In [ ]:
data, raw_mean, raw_std = load_and_normalize(csv_path)
data.shape

## 3) PCA attractor projection

Project the normalized 5D trajectory into PCA space with NumPy SVD and save 2D/3D attractor plots.

In [ ]:
projected, components, explained_variance_ratio = compute_pca(data)
save_pca_attractor_2d(projected, output_dir)
save_pca_attractor_3d(projected, output_dir)
explained_variance_ratio

## 4) Poincaré section

Use the hyperplane `PC1 = median(PC1)` and keep only positive-slope crossings to inspect the section in `(PC2, PC3)`.

In [ ]:
poincare_points, threshold = find_poincare_crossings(projected)
save_poincare_section(poincare_points, threshold, output_dir)
poincare_points.shape

## 5) Largest Lyapunov exponent

Apply Rosenstein's algorithm directly in the 5D state space with a temporal exclusion window of 10 samples and a 20-step divergence horizon.

In [ ]:
lle, steps, mean_log_divergence, fit_line = rosenstein_lle(data, temporal_window=10, n_steps=20, fit_range=(1, 10))
save_lyapunov_plot(steps, mean_log_divergence, fit_line, lle, output_dir)
lle_interpretation = interpret_lle(lle)
lle, lle_interpretation

## 6) Recurrence plot and RQA

Build the recurrence plot in the original 5D space using the 10th percentile distance threshold, then compute the requested RQA metrics.

In [ ]:
recurrence, epsilon = build_recurrence_matrix(data)
save_recurrence_plot(recurrence, output_dir)
rqa_metrics = compute_rqa_metrics(recurrence, min_line_length=2)
epsilon, rqa_metrics

## 7) Chaos interpretation summary

Combine the LLE and recurrence metrics into the same structured report as the command-line script. Positive LLE with lower DET suggests chaos, while near-zero/negative LLE and high DET suggest periodic or stable dynamics.

In [ ]:
verdict = synthesize_verdict(lle, rqa_metrics)
results = AnalysisResults5D(
    dataset=os.path.basename(csv_path),
    n_time_steps=data.shape[0],
    explained_variance_ratio=explained_variance_ratio,
    lle=lle,
    lle_interpretation=lle_interpretation,
    rqa_metrics=rqa_metrics,
    verdict=verdict,
)
print_summary(results)